In [ ]:
## Build Dataset Structure for MM5 Tri-modal Data

import os
import shutil
import pandas as pd

# Define source (raw download) and target (your clean structure)
RAW_DIR = "data/MM5_ALIGNED/"
TARGET_DIR = "dataset/MM5/"

# Folder mapping: Target -> Source
TARGET_FOLDERS = {
    "RGB": "RGB3",
    "Depth": "D16",
    "Thermal": "T16",
    "Class_Annotations": "ANNO_CLASS"
}

def setup_directories():
    for target in TARGET_FOLDERS.keys():
        os.makedirs(os.path.join(TARGET_DIR, target), exist_ok=True)

def migrate_data():
    # Read the CSVs expecting ID numbers instead of full filenames
    train_df = pd.read_csv(os.path.join(RAW_DIR, "train_dataset.csv"), header=0, names=["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"])
    eval_df = pd.read_csv(os.path.join(RAW_DIR, "eval_dataset.csv"), header=0, names=["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"])

    # Combine all IDs into a single list
    all_ids = pd.concat([train_df, eval_df])["ID"].tolist()

    print(f"Found {len(all_ids)} total image IDs to migrate.")

    missing_files = []

    for img_id in all_ids:
        # Construct the filename strictly from the ID
        filename = f"{str(img_id).strip()}.png"

        for target_folder, source_folder in TARGET_FOLDERS.items():
            src_path = os.path.join(RAW_DIR, source_folder, filename)
            tgt_path = os.path.join(TARGET_DIR, target_folder, filename)
            print(f"Processing ID {img_id}: {src_path} -> {tgt_path}")
            
            if os.path.exists(src_path):
                shutil.copy2(src_path, tgt_path)
            else:
                missing_files.append(src_path)

    if missing_files:
        print(f"WARNING: Could not find {len(missing_files)} files. Showing first 5:")
        for f in missing_files[:5]:
            print(f" - {f}")
    else:
        print("All files migrated successfully.")


print("Initializing clean tri-modal dataset structure...")
setup_directories()
migrate_data()
print("Migration complete. The dataset/MM5/ directory is ready.")

In [ ]:
## Visualizer for MM5 Tri-modal Dataset

import os
import cv2
import numpy as np
import pandas as pd

# Define paths matching our established directory structure
DATASET_DIR = "dataset/MM5"
TRAIN_CSV_FILE = os.path.join(DATASET_DIR, "train_dataset.csv")  
EVAL_CSV_FILE = os.path.join(DATASET_DIR, "eval_dataset.csv")

def prepare_for_display(img, title, colormap=None):
    """
    Normalizes 16-bit or low-variance 8-bit arrays to 0-255 for screen display,
    applies an optional colormap, and burns the title label into the image.
    """
    # Handle missing or corrupted files gracefully
    if img is None:
        img = np.zeros((480, 640, 3), dtype=np.uint8)
        cv2.putText(img, "FILE MISSING", (50, 240), cv2.FONT_HERSHEY_SIMPLEX, 2, (0, 0, 255), 3)
        return img

    # Min-Max normalize to 0-255 for visual contrast
    img_norm = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

    # Apply colormap or convert grayscale to 3-channel BGR so it stacks cleanly with RGB
    if colormap is not None:
        img_bgr = cv2.applyColorMap(img_norm, colormap)
    elif len(img_norm.shape) == 2:
        img_bgr = cv2.cvtColor(img_norm, cv2.COLOR_GRAY2BGR)
    else:
        img_bgr = img_norm

    # Ensure uniform sizing for the 2x2 grid
    img_resized = cv2.resize(img_bgr, (640, 480))

    # Add a semi-transparent black background for the text label to ensure readability
    overlay = img_resized.copy()
    cv2.rectangle(overlay, (0, 0), (220, 50), (0, 0, 0), -1)
    img_resized = cv2.addWeighted(overlay, 0.6, img_resized, 0.4, 0)

    # Burn the text label into the top-left corner
    cv2.putText(img_resized, title, (15, 35), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)
    
    return img_resized

def run_visualizer():
    # Read the dataset IDs
    if not os.path.exists(TRAIN_CSV_FILE):
        print(f"Error: {TRAIN_CSV_FILE} not found. Please ensure it is in the same directory.")
        return

    if not os.path.exists(EVAL_CSV_FILE):
        print(f"Error: {EVAL_CSV_FILE} not found. Please ensure it is in the same directory.")
        return

    train_df = pd.read_csv(TRAIN_CSV_FILE, header=0, names=["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"])
    eval_df = pd.read_csv(EVAL_CSV_FILE, header=0, names=["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"])
    ids = pd.concat([train_df, eval_df])["ID"].tolist()
    
    print(f"Loaded {len(ids)} images for visualization.")
    print("Controls:\n - Press <SPACE> for the next set of images.\n - Press <Q> to quit.")

    # Create a named, resizable window
    window_name = "MM5 Tri-Modal Viewer"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    for img_id in ids:
        filename = f"{str(img_id).strip()}.png"
        
        # Load raw images
        rgb = cv2.imread(os.path.join(DATASET_DIR, "RGB", filename))
        depth = cv2.imread(os.path.join(DATASET_DIR, "Depth", filename), cv2.IMREAD_UNCHANGED)
        thermal = cv2.imread(os.path.join(DATASET_DIR, "Thermal", filename), cv2.IMREAD_UNCHANGED)
        mask = cv2.imread(os.path.join(DATASET_DIR, "Class_Annotations", filename), cv2.IMREAD_UNCHANGED)

        # Prepare each quadrant with specific colormaps for visual clarity
        q1_rgb = prepare_for_display(rgb, "RGB")
        q2_depth = prepare_for_display(depth, "Depth (16-bit)", cv2.COLORMAP_VIRIDIS)
        q3_thermal = prepare_for_display(thermal, "Thermal (16-bit)", cv2.COLORMAP_INFERNO)
        
        # The semantic mask is multiplied by a colormap to separate distinct integer IDs
        q4_mask = prepare_for_display(mask, "Class Annotations", cv2.COLORMAP_TWILIGHT_SHIFTED)

        # Construct the 2x2 grid using numpy stacking
        top_row = np.hstack((q1_rgb, q2_depth))
        bottom_row = np.hstack((q3_thermal, q4_mask))
        grid = np.vstack((top_row, bottom_row))

        # Display the grid
        cv2.imshow(window_name, grid)

        # Keyboard event loop
        while True:
            key = cv2.waitKey(0) & 0xFF
            
            # ASCII 32 is the <SPACE> bar
            if key == 32:
                break
            # ASCII 113 is 'q'
            elif key == ord('q'):
                print("Viewer terminated by user.")
                cv2.destroyAllWindows()
                return

    cv2.destroyAllWindows()
    print("Reached the end of the dataset.")

run_visualizer()

Phase 1: Data Versioning with DVC
Before writing any PyTorch code, the raw MM5 dataset must be frozen. DVC allows you to version massive 16-bit image directories without bloating your Git history.

```bash
# Initialize DVC in your repository
dvc init

# Track the raw dataset
dvc add dataset/MM5/

# Git track the DVC metadata files (not the images)
git add dataset/.gitignore dataset/MM5.dvc
git commit -m "chore: track raw MM5 dataset with DVC"
```

Phase 2: Curation and Analysis with FiftyOne
FiftyOne is exceptional for multimodal data. We will use it to programmatically filter out the UV and NIR channels, structure our Train/Val/Test splits, and visually inspect the alignment of the ground truth segmentation masks.

In [ ]:
## FiftyOne Dataset Ingestion for MM5 Tri-modal Data

import os
import json
import cv2
import numpy as np
import pandas as pd
import fiftyone as fo
from PIL import Image

DATASET_DIR = "dataset/MM5"
PROXY_DIR = "dataset/MM5_Proxies"
MASK_MULTIPLIER = 8  # The visual scaling factor for the UI

def load_json_mapping():
    """Parses label_mapping.json to guarantee exact integer-to-string matching."""
    json_path = os.path.join(DATASET_DIR, "label_mapping.json")
    if not os.path.exists(json_path):
        print(f"Error: {json_path} not found. Masks will not render correctly.")
        return {}
        
    with open(json_path, 'r') as f:
        data = json.load(f)
        
    mapping = {}
    if "categories" in data:
        for cat in data["categories"]:
            mapping[int(cat["id"])] = cat["name"]
    else:
        for k, v in data.items():
            if isinstance(v, int) or (isinstance(v, str) and v.isdigit()):
                mapping[int(v)] = str(k)
            elif isinstance(k, int) or (isinstance(k, str) and k.isdigit()):
                mapping[int(k)] = str(v)
            
    print(f"Loaded {len(mapping)} classes from label_mapping.json")
    return mapping

def create_ui_proxy(raw_path, filename, modality, mapping=None):
    """Generates 8-bit UI proxies. Applies visual scaling to masks."""
    if not os.path.exists(raw_path):
        return None, None
        
    modality_dir = os.path.join(PROXY_DIR, modality)
    os.makedirs(modality_dir, exist_ok=True)
    proxy_path = os.path.join(modality_dir, filename)
    
    if modality == "Class_Annotations":
        # 1. Load via PIL to extract the true integer IDs
        img = Image.open(raw_path)
        img_arr = np.array(img)
        
        if len(img_arr.shape) > 2:
            img_arr = img_arr[:, :, 0]
            
        img_arr = img_arr.astype(np.uint8)
        unique_ids = np.unique(img_arr).tolist()
        
        # 2. Extract string names based on the ORIGINAL IDs
        present_classes = []
        if mapping:
            present_classes = [mapping[uid] for uid in unique_ids if uid in mapping and mapping[uid].lower() != "background"]
            
        # 3. Apply the Visual Scaling Factor (Multiply by 8)
        # We use clip to ensure we don't overflow the 8-bit 255 limit
        visual_img_arr = np.clip(img_arr.astype(np.uint16) * MASK_MULTIPLIER, 0, 255).astype(np.uint8)
            
        if not os.path.exists(proxy_path):
            cv2.imwrite(proxy_path, visual_img_arr)
            
        return proxy_path, present_classes

    # Return cached proxy if it exists for sensor feeds
    if os.path.exists(proxy_path):
        return proxy_path, None

    if modality == "RGB":
        img = cv2.imread(raw_path)
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        cv2.imwrite(proxy_path, img)
    elif modality == "Depth":
        img = cv2.imread(raw_path, cv2.IMREAD_UNCHANGED)
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        img = cv2.applyColorMap(img, cv2.COLORMAP_VIRIDIS)
        cv2.imwrite(proxy_path, img)
    elif modality == "Thermal":
        img = cv2.imread(raw_path, cv2.IMREAD_UNCHANGED)
        img = cv2.normalize(img, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
        img = cv2.applyColorMap(img, cv2.COLORMAP_INFERNO)
        cv2.imwrite(proxy_path, img)
        
    return proxy_path, None

def create_fiftyone_dataset():
    dataset_name = "MM5-TriModal-Structural-POC"
    if dataset_name in fo.list_datasets():
        fo.delete_dataset(dataset_name)
    
    dataset = fo.Dataset(name=dataset_name)
    dataset.persistent = True
    
    # --- The Dictionary Shift ---
    # Shift the target dictionary keys to match the multiplied proxy pixels
    raw_mask_targets = load_json_mapping()
    if raw_mask_targets:
        visual_mask_targets = {k * MASK_MULTIPLIER: v for k, v in raw_mask_targets.items()}
        dataset.default_mask_targets = visual_mask_targets
        
    dataset.add_group_field("group", default="rgb")

    splits = {
        "Train": os.path.join(DATASET_DIR, "train_dataset.csv"),
        "Eval": os.path.join(DATASET_DIR, "eval_dataset.csv")
    }
    csv_columns = ["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"]
    
    samples = []
    
    for split_tag, csv_path in splits.items():
        if not os.path.exists(csv_path):
            continue
            
        df = pd.read_csv(csv_path, header=0, names=csv_columns)
        img_ids = df["ID"].tolist()
        
        print(f"Compiling {split_tag} split and applying visual mask scaling...")
        for img_id in img_ids:
            filename = f"{str(img_id).strip()}.png"
            
            raw_rgb = os.path.join(DATASET_DIR, "RGB", filename)
            raw_depth = os.path.join(DATASET_DIR, "Depth", filename)
            raw_thermal = os.path.join(DATASET_DIR, "Thermal", filename)
            raw_mask = os.path.join(DATASET_DIR, "Class_Annotations", filename)
            
            if not os.path.exists(raw_rgb): continue

            # Generate Proxies
            proxy_rgb, _ = create_ui_proxy(raw_rgb, filename, "RGB")
            proxy_depth, _ = create_ui_proxy(raw_depth, filename, "Depth")
            proxy_thermal, _ = create_ui_proxy(raw_thermal, filename, "Thermal")
            # The mask returned here has been multiplied by MASK_MULTIPLIER
            proxy_mask, present_classes = create_ui_proxy(raw_mask, filename, "Class_Annotations", mapping=raw_mask_targets)

            scene_labels = None
            if present_classes:
                scene_labels = fo.Classifications(
                    classifications=[fo.Classification(label=cls) for cls in present_classes]
                )

            scene_group = fo.Group()
            
            # --- RGB Sample ---
            sample_rgb = fo.Sample(filepath=proxy_rgb)
            sample_rgb["group"] = scene_group.element("rgb")
            sample_rgb.tags.append(split_tag)
            
            # --- Depth Sample ---
            if proxy_depth:
                sample_depth = fo.Sample(filepath=proxy_depth)
                sample_depth["group"] = scene_group.element("depth")
                sample_depth.tags.append(split_tag)
                
            # --- Thermal Sample ---
            if proxy_thermal:
                sample_thermal = fo.Sample(filepath=proxy_thermal)
                sample_thermal["group"] = scene_group.element("thermal")
                sample_thermal.tags.append(split_tag)
                
            # --- Attach Native Labels and Scaled Masks ---
            if proxy_mask:
                seg_mask = fo.Segmentation(mask_path=proxy_mask)
                
                sample_rgb["ground_truth"] = seg_mask
                if scene_labels: sample_rgb["scene_classes"] = scene_labels
                
                if 'sample_depth' in locals():
                    sample_depth["ground_truth"] = seg_mask
                    if scene_labels: sample_depth["scene_classes"] = scene_labels
                
                if 'sample_thermal' in locals():
                    sample_thermal["ground_truth"] = seg_mask
                    if scene_labels: sample_thermal["scene_classes"] = scene_labels

            samples.append(sample_rgb)
            if 'sample_depth' in locals(): samples.append(sample_depth)
            if 'sample_thermal' in locals(): samples.append(sample_thermal)

            if 'sample_depth' in locals(): del sample_depth
            if 'sample_thermal' in locals(): del sample_thermal
            
    dataset.add_samples(samples)
    print(f"\nIngestion Complete.")
    
    session = fo.launch_app(dataset)
    session.wait()

create_fiftyone_dataset()

Phase 3: The PyTorch DataLoaderBecause MM5 utilizes 16-bit TIFFs/PNGs for depth and thermal, standard PIL.Image or torchvision.transforms will clip or destroy the radiometric data. We must use OpenCV to read the raw arrays, apply dataset-specific modality-wise normalization, and stack them into the final tensor.  

In [ ]:
## Tri-Modal Segmentation Dataset Loader for PyTorch

import os
import cv2
import torch
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader

class TriModalSegDataset(Dataset):
    def __init__(self, csv_file, root_dir="dataset/MM5", target_size=(480, 640)):
        """
        Args:
            csv_file (str): Full path to 'train_dataset.csv' or 'eval_dataset.csv'.
            root_dir (str): Directory containing RGB, Depth, Thermal, and Class_Annotations.
        """
        self.data_frame = pd.read_csv(csv_file, header=0, names=["ID", "Sequence", "Category", "Subcategory", "Challenges", "Category_Subcategory"])
        self.root_dir = root_dir
        self.target_size = target_size

    def __len__(self):
        return len(self.data_frame)

    def __getitem__(self, idx):
        img_id = str(self.data_frame.iloc[idx, 0]).strip()
        filename = f"{img_id}.png"

        # 1. Load RGB (8-bit, 3 channels)
        rgb_path = os.path.join(self.root_dir, "RGB", filename)
        rgb = cv2.imread(rgb_path)
        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)
        rgb = cv2.resize(rgb, (self.target_size[1], self.target_size[0]))
        rgb = rgb.astype(np.float32) / 255.0

        # 2. Load Depth (16-bit, 1 channel)
        depth_path = os.path.join(self.root_dir, "Depth", filename)
        depth = cv2.imread(depth_path, cv2.IMREAD_UNCHANGED)
        depth = cv2.resize(depth, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_NEAREST)
        depth = depth.astype(np.float32)
        depth_min, depth_max = depth.min(), depth.max()
        if depth_max > depth_min:
            depth = (depth - depth_min) / (depth_max - depth_min)

        # 3. Load Thermal LWIR (16-bit, 1 channel)
        thermal_path = os.path.join(self.root_dir, "Thermal", filename)
        thermal = cv2.imread(thermal_path, cv2.IMREAD_UNCHANGED)
        thermal = cv2.resize(thermal, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_NEAREST)
        thermal = thermal.astype(np.float32)
        thermal_min, thermal_max = thermal.min(), thermal.max()
        if thermal_max > thermal_min:
            thermal = (thermal - thermal_min) / (thermal_max - thermal_min)

        # 4. Load Semantic Mask
        mask_path = os.path.join(self.root_dir, "Class_Annotations", filename)
        mask = cv2.imread(mask_path, cv2.IMREAD_UNCHANGED)
        mask = cv2.resize(mask, (self.target_size[1], self.target_size[0]), interpolation=cv2.INTER_NEAREST)
        
        # 5. Stack and convert to PyTorch standard [C, H, W]
        rgbdt = np.dstack([rgb, depth, thermal])
        tensor_rgbdt = torch.from_numpy(rgbdt).permute(2, 0, 1).float()
        tensor_mask = torch.from_numpy(mask).long()

        return tensor_rgbdt, tensor_mask

TRAIN_CSV = "dataset/MM5/train_dataset.csv"

if os.path.exists(TRAIN_CSV):
    train_dataset = TriModalSegDataset(csv_file=TRAIN_CSV)
    train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=4, pin_memory=True)
    print("DataLoader successfully reading from updated CSV layout location.")

Phase 4: Adapting a YOLO Architecture for 5 Channels
YOLOv8-Seg utilizes a CSPDarknet backbone designed strictly for 3-channel RGB input. To utilize this architecture for a 5-channel tensor, we must surgically modify the very first convolutional "stem" layer of the network before initiating the training loop.

In [ ]:
## Adapting YOLO-style Segmentation Backbone for 5-Channel Input

import torch.nn as nn
# Assuming `model` is your YOLO-style segmentation backbone
# The default first layer: nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1)

# Overwrite the stem to accept 5 input channels
original_conv = model.backbone.stem.conv
model.backbone.stem.conv = nn.Conv2d(
    in_channels=5, 
    out_channels=original_conv.out_channels, 
    kernel_size=original_conv.kernel_size, 
    stride=original_conv.stride, 
    padding=original_conv.padding,
    bias=False
)

# Initialize the new weights. (Optional but recommended: copy the pre-trained RGB 
# weights into the first 3 channels, and use Kaiming initialization for channels 4 & 5).

Phase 5: Instrumentation (TensorBoard & GradCAM)
With the DataLoader and model architecture aligned, the training loop requires instrumentation to evaluate the multimodal fusion.

1. TensorBoard:
Integrate PyTorch's SummaryWriter directly into your epoch loop. This allows you to track real-time learning curves, learning rate decay, and hyperparameter grids across different ablation runs (e.g., comparing your 5-channel model against a 3-channel RGB-only baseline).

2. Multimodal GradCAM:
GradCAM is vital for proving that the AI is actually using the depth and thermal data, rather than just relying on the RGB channels.

Because standard GradCAM implementations are built for RGB imagery, you will apply the hook to the final convolutional layer of the YOLO backbone.

When the heatmap is generated, you will overlay it strictly onto the RGB slice tensor_rgbdt[0:3, :, :] for human visual evaluation, or overlay it onto the thermal slice tensor_rgbdt[4, :, :] to prove the network is attending to high-contrast thermal signatures.

## Extract "best" hyperparameters from HPO trials

In [1]:
import optuna
study = optuna.load_study(study_name="trimodal_research_sweep", storage="sqlite:///trimodal_research_sweep.db")
print(study.best_params)

{'gamma': 1.6626811271477564, 'dice_weight': 0.6250057441774319, 'optimizer': 'SGD', 'lr': 0.0752684692189, 'weight_decay': 0.0003049896403468519, 'scheduler': 'CosineAnnealing', 'sgd_momentum': 0.9684739916475875, 'nesterov': True}
